# Palmer Penguins: Exploratory Data Analysis and Data Preparation

## Project goal

This notebook explores and prepares the Seaborn Palmer Penguins dataset
for use in a Django web application called Penguin Explorer.

The final application will allow users to browse, search, filter, and
view individual penguin records.

## Workflow

1. Load the original Seaborn dataset.
2. Export an untouched raw copy.
3. Inspect its structure, data types, and missing values.
4. Explore distributions and relationships using visualisations.
5. Clean the data and document every decision.
6. Export a processed CSV for import into Django.

## 1. Load and preserve the raw dataset

The Palmer Penguins dataset is loaded from Seaborn. Before carrying out
any exploration or cleaning, an unchanged copy is exported to the
`data/raw` folder. This preserves the original source data and makes
the preparation process reproducible.

In [2]:
from pathlib import Path
import seaborn as sns

BASE_DIR = Path.cwd().parent

raw_data_folder = BASE_DIR / "data" / "raw"
raw_data_folder.mkdir(parents=True, exist_ok=True)

raw_data_path = raw_data_folder / "penguins_raw.csv"

penguins = sns.load_dataset("penguins")
penguins.to_csv(raw_data_path, index=False)

print(raw_data_path)
penguins.head()

/Users/phekomantlhasi/my_django_projects/penguin_explorer/data/raw/penguins_raw.csv


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


## 2. Load the raw dataset

The dataset is now loaded from the raw CSV file rather than directly from
Seaborn. This ensures that all exploration and cleaning steps use the same
preserved source file.

In [3]:
raw_data_path = BASE_DIR / 'data' / 'raw' / 'penguins_raw.csv'

penguins = pd.read_csv(raw_data_path)

penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


## 3. Initial data inspection

This section examines the dataset's size, columns, data types, missing values,
and duplicate records before any cleaning decisions are made.

In [4]:
penguins.shape

(344, 7)

In [10]:
penguins.columns

Index(['species', 'island', 'bill_length_mm', 'bill_depth_mm',
       'flipper_length_mm', 'body_mass_g', 'sex'],
      dtype='str')

### Dataset shape columns

The raw Penguins dataset contains **344 rows** and **7 columns**.

Each row represents one penguin observation, while the columns contain
information such as species, island, physical measurements, sex, and ye

In [5]:
penguins.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
dtypes: float64(4), str(3)
memory usage: 18.9 KB


### Dataset infomation

The dataset contains 7 columns: `species`, `island`, `bill_length_mm`,
`bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, and `sex`.

Initial inspection shows that the measurement columns
(`bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, and `body_mass_g`)
and the `sex` column contain missing values. These missing values will be
examined further before making any cleaning decisions.

In [6]:
penguins.describe()

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
count,342.000000,342.000000,342.000000,342.000000
mean,43.921930,17.151170,200.915205,4201.754386
std,5.459584,1.974793,14.061714,801.954536
min,32.100000,13.100000,172.000000,2700.000000
25%,39.225000,15.600000,190.000000,3550.000000
50%,44.450000,17.300000,197.000000,4050.000000
75%,48.500000,18.700000,213.000000,4750.000000
max,59.600000,21.500000,231.000000,6300.000000


### Summary statistics

This output provides summary statistics for each numeric column in the dataset.

It includes the number of non-missing values, mean, standard deviation,
minimum, maximum, and quartile values. These statistics help identify the
typical range and spread of each penguin measurement.

In [8]:
penguins.isna().sum()

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

In [15]:
penguins[penguins.isna().any(axis=1)]

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
8,Adelie,Torgersen,34.1,18.1,193.0,3475.0,NaN
9,Adelie,Torgersen,42.0,20.2,190.0,4250.0,NaN
10,Adelie,Torgersen,37.8,17.1,186.0,3300.0,NaN
11,Adelie,Torgersen,37.8,17.3,180.0,3700.0,NaN
47,Adelie,Dream,37.5,18.9,179.0,2975.0,NaN
246,Gentoo,Biscoe,44.5,14.3,216.0,4100.0,NaN
286,Gentoo,Biscoe,46.2,14.4,214.0,4650.0,NaN
324,Gentoo,Biscoe,47.3,13.8,216.0,4725.0,NaN
336,Gentoo,Biscoe,44.5,15.7,217.0,4875.0,NaN


### Dataset missing values

As previously detected, there were missing values observed in the columns of the datasets. This particular inspection measures how many missing values there are for each column containing missing values.

In [23]:
penguins.duplicated().sum()

np.int64(0)

### Dataset duplicates inspection

Initial inspection shows that there are no duplicated values in the dataset which can inflate the data leading to incorrect decision making.

## 4. Data cleaning

Two records contain no physical measurement data and cannot provide useful
penguin profiles for the web application. These records will be removed.

Some records have a missing `sex` value. These records will be retained because
their other data is valid, and the missing value will be displayed as
“Not recorded” in the Django application rather than guessed or replaced.

In [24]:
measurement_columns = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
]

penguins_cleaned = penguins.dropna(subset=measurement_columns, how='all')
penguins_cleaned

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male
...,...,...,...,...,...,...,...
338,Gentoo,Biscoe,47.2,13.7,214.0,4925.0,Female
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


## 5. Export the processed dataset

After removing records with no physical measurement data, the cleaned dataset
contains 342 records. Missing values in the `sex` column have been retained and
will be handled as optional values in the Django application.

The cleaned dataset is exported to the `data/processed` folder for later
import into the Django database.

In [26]:
processed_data_folder = BASE_DIR / 'data'/ 'processed'
processed_data_folder.mkdir(parents=True, exist_ok=True)

processed_data_path = processed_data_folder / 'penguins_clean.csv'

penguins_cleaned.to_csv(processed_data_path, index=False)

print(f"Cleaned dataset saved to: {processed_data_path}")


Cleaned dataset saved to: /Users/phekomantlhasi/my_django_projects/penguin_explorer/data/processed/penguins_clean.csv


### Processed dataset exported

The cleaned dataset was exported as `penguins_clean.csv` in the
`data/processed` folder. The exported file excludes the two records with no
physical measurement data while retaining records where `sex` was not recorded.

This processed CSV will be used to populate the Django application's database.